# Credit Risk EDA

Dataset: Credit Risk Benchmark Dataset
Target: `dlq_2yrs` (1 = delinquent in 2 years)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_PATH = '../data/raw/Credit Risk Benchmark Dataset.csv'
df = pd.read_csv(DATA_PATH)
print('Shape:', df.shape)
df.head()

## Data Quality Check

In [ ]:
df.info()

In [ ]:
missing = df.isnull().sum().rename('missing')
pct_missing = (missing / len(df) * 100).rename('pct_missing')
pd.concat([missing, pct_missing, df.dtypes.rename('dtype')], axis=1)

## Target Distribution

In [ ]:
target_counts = df['dlq_2yrs'].value_counts().sort_index()
target_counts

In [ ]:
def plot_target_distribution(counts):
    labels = ['Not Delinquent (0)', 'Delinquent (1)']
    fig, ax = plt.subplots(figsize=(6, 4))
    bars = ax.bar(labels, counts.values, color=['steelblue', 'tomato'])
    ax.bar_label(bars)
    ax.set_title('Target Distribution')
    ax.set_ylabel('Count')
    plt.tight_layout()
    plt.show()

In [ ]:
plot_target_distribution(target_counts)

## Numeric Summary

In [ ]:
df.describe().T

## Feature Histograms

In [ ]:
features = [c for c in df.columns if c != 'dlq_2yrs']
n_cols = 3
n_rows = -(-len(features) // n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4 * n_rows))
axes = axes.flatten()

for i, col in enumerate(features):
    axes[i].hist(df[col], bins=40, color='steelblue', alpha=0.8)
    axes[i].set_title(col)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# Remove rows with invalid values in 'rev_util' and 'debt_ratio'
mask = df['rev_util'].between(0, 1) & df['debt_ratio'].between(0, 1)

rows_before = len(df)
df = df.loc[mask].copy()
rows_after = len(df)

print(f"Rows before: {rows_before}")
print(f"Rows after:  {rows_after}")
print(f"Removed:     {rows_before - rows_after}")

In [ ]:
# Remove outliers in 'monthly_inc', 'real_estate' and 'open_credit' using IQR method
for col in ['monthly_inc', 'real_estate', 'open_credit']:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    mask = (df[col] >= lower_bound) & (df[col] <= upper_bound)
    rows_before = len(df)
    df = df.loc[mask].copy()
    rows_after = len(df)
    
    print(f"{col} - Rows before: {rows_before}, after: {rows_after}, removed: {rows_before - rows_after}")

In [ ]:
# Plot the cleaned data
features = [c for c in df.columns if c != 'dlq_2yrs']
n_cols = 3
n_rows = -(-len(features) // n_cols)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4 * n_rows))
axes = axes.flatten()
for i, col in enumerate(features):
    axes[i].hist(df[col], bins=40, color='steelblue', alpha=0.8)
    axes[i].set_title(col)
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
# investigate late_30_59, late_60_89, late_90 columns for potential data quality issues
late_cols = ['late_30_59', 'late_60_89', 'late_90']
for col in late_cols:
    counts = df[col].value_counts()
    print(f"{col} value counts:\n{counts}\n")

In [ ]:
# Crate new column 'ever_late' and 'long_late' to capture if a borrower was ever late and how late they were
df['ever_late'] = ((df['late_30_59'] == 1) | (df['late_60_89'] == 1) | (df['late_90'] == 1)).astype(int)
df['long_late'] = (df['late_90'] == 1).astype(int)

# investigate the new columns
print("ever_late value counts:\n", df['ever_late'].value_counts())
print("\nlong_late value counts:\n", df['long_late'].value_counts())

In [ ]:
# Remove late_30_59, late_60_89, late_90 columns as they are now redundant
df.drop(columns=late_cols, inplace=True)    

In [ ]:
# Check the target distribution again after cleaning
target_counts = df['dlq_2yrs'].value_counts().sort_index()
plot_target_distribution(target_counts)

## Correlation Matrix

In [ ]:
# Calculate correlation to dlq_2yrs
correlations = df.corr()['dlq_2yrs'].sort_values(ascending=False)
print("Correlation to dlq_2yrs:\n", correlations)

In [ ]:
# Sort columns by absolute correlation to dlq_2yrs
abs_correlations = correlations.abs().sort_values(ascending=False)
sorted_cols = abs_correlations.index.tolist()
print("\nColumns sorted by absolute correlation to dlq_2yrs:\n", sorted_cols)

In [ ]:
# Plot correlation matrix by sorted column order showing number on the heatmap
corr = df.corr(numeric_only=True)
corr = corr.loc[sorted_cols, sorted_cols]

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
fig.colorbar(im, ax=ax)
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=45, ha='right')
ax.set_yticklabels(corr.columns)
# Add correlation values on the heatmap
for i in range(len(corr.columns)):
    for j in range(len(corr.columns)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha='center', va='center', color='black')
ax.set_title('Correlation Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# Keep useful features for modeling
useful_features = ['rev_util', 'ever_late', 'monthly_inc', 'age']
df_model = df[useful_features + ['dlq_2yrs']].copy()
print("Features kept for modeling:\n", df_model.columns.tolist())

## Model Traing

In [ ]:
# Train/test split
X = df_model.drop(columns='dlq_2yrs')
y = df_model['dlq_2yrs']

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Training set shape: {X_train.shape}, Testing set shape: {X_test.shape}")

In [ ]:
# Plot y_train distribution
train_counts = y_train.value_counts().sort_index()
plot_target_distribution(train_counts)

In [ ]:
# balancing training data using undersampling
from imblearn.under_sampling import RandomUnderSampler
rus = RandomUnderSampler(random_state=42)
X_train_balanced, y_train_balanced = rus.fit_resample(X_train, y_train)
print(f"Balanced training set shape: {X_train_balanced.shape}") 

In [ ]:
X_train, y_train = X_train_balanced, y_train_balanced
train_counts = y_train.value_counts().sort_index()
plot_target_distribution(train_counts)

In [ ]:
test_counts = y_test.value_counts().sort_index()
plot_target_distribution(test_counts)

### Logrithmic Regression Model

In [ ]:
from sklearn.linear_model import LogisticRegression
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test) 

#Print classification report
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred_lr, target_names=['Not Delinquent (0)', 'Delinquent (1)']))

In [ ]:
# Plot confusion matrix
from sklearn.metrics import confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay

cm_lr = confusion_matrix(y_test, y_pred_lr)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_lr,
    display_labels=['Not Delinquent (0)', 'Delinquent (1)']
)
fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(cmap='Blues', ax=ax, colorbar=False, values_format='d')
ax.set_title('Confusion Matrix for Logistic Regression\nRows: Actual, Columns: Predicted')
plt.tight_layout()
plt.show()

In [ ]:
# Function to calculate accept rate and bad rate
def calculate_rates(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    accept_rate = (tn + fn) / len(y_true)
    bad_rate = fn / (fn + tn) if (fn + tn) > 0 else 0
    return accept_rate, bad_rate

In [ ]:
# Calculate accept rate and bad rate based on Logistic Regression predictions
accept_rate_lr, bad_rate_lr = calculate_rates(y_test, y_pred_lr)
print(f"Logistic Regression - Accept Rate: {accept_rate_lr:.2%}, Bad Rate: {bad_rate_lr:.2%}")

In [ ]:
y_pred_lr_train = lr.predict(X_train)
accept_rate_lr_train, bad_rate_lr_train = calculate_rates(y_train, y_pred_lr_train)
print(f"Logistic Regression (Train) - Accept Rate: {accept_rate_lr_train:.2%}, Bad Rate: {bad_rate_lr_train:.2%}")  

In [ ]:
# Apply perdiction on X
y_pred_lr_all = lr.predict(X)
accept_rate_lr_all, bad_rate_lr_all = calculate_rates(y, y_pred_lr_all)
print(f"Logistic Regression (All) - Accept Rate: {accept_rate_lr_all:.2%}, Bad Rate: {bad_rate_lr_all:.2%}")  

In [ ]:
# Plot logistic regression coefficients
coef_df = pd.DataFrame({
    'feature': X_train.columns,
    'coefficient': lr.coef_[0]
}).sort_values(by='coefficient', key=abs, ascending=False)

import seaborn as sns

# Plot the coefficients
plt.figure(figsize=(10, 6))
sns.barplot(x='coefficient', y='feature', data=coef_df)
plt.title('Logistic Regression Coefficients')
plt.show()

In [ ]:
# cross-validation with 5 folds
from sklearn.model_selection import cross_val_score
cv_scores = cross_val_score(lr, X_train, y_train, cv=5, scoring='f1')
print("Cross-validation F1 scores:", cv_scores)
print("Average CV F1 score:", cv_scores.mean()) 

In [ ]:
# grid search with cross-validation for logistic regression
from sklearn.model_selection import GridSearchCV
param_grid = {
    'C': [0.01, 0.1, 1, 10, 100],
    'l1_ratio': [0, 1],
    'solver': ['liblinear']
}


grid_search = GridSearchCV(LogisticRegression(max_iter=1000, random_state=42), param_grid, cv=5, scoring='f1')
grid_search.fit(X_train, y_train)
print("Best parameters:", grid_search.best_params_)
print("Best CV F1 score:", grid_search.best_score_) 

### Xgboost

In [ ]:
from xgboost import XGBClassifier
xgb = XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss')
xgb.fit(X_train, y_train)
y_pred_xgb = xgb.predict(X_test)    

#Print classification report for XGBoost
print(classification_report(y_test, y_pred_xgb, target_names=['Not Delinquent (0)', 'Delinquent (1)']))


In [ ]:
# Plot confusion matrix for XGBoost (rows = Actual, cols = Predicted)
cm_xgb = confusion_matrix(y_test, y_pred_xgb)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_xgb,
    display_labels=['Not Delinquent (0)', 'Delinquent (1)']
)
fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(cmap='Blues', ax=ax, colorbar=False, values_format='d')
ax.set_title('Confusion Matrix for XGBoost\nRows: Actual, Columns: Predicted')
plt.tight_layout()
plt.show()

In [ ]:
# calculate accept rate and bad rate based on XGBoost predictions
accept_rate_xgb, bad_rate_xgb = calculate_rates(y_test, y_pred_xgb)
print(f"XGBoost - Accept Rate: {accept_rate_xgb:.2%}, Bad Rate: {bad_rate_xgb:.2%}")

In [ ]:
y_pred_xgb_train = xgb.predict(X_train)
accept_rate_xgb_train, bad_rate_xgb_train = calculate_rates(y_train, y_pred_xgb_train)
print(f"XGBoost (Train) - Accept Rate: {accept_rate_xgb_train:.2%}, Bad Rate: {bad_rate_xgb_train:.2%}")    

In [ ]:
y_pred_xgb_all = xgb.predict(X)
accept_rate_xgb_all, bad_rate_xgb_all = calculate_rates(y, y_pred_xgb_all)
print(f"XGBoost (All) - Accept Rate: {accept_rate_xgb_all:.2%}, Bad Rate: {bad_rate_xgb_all:.2%}")

In [ ]:
# Compare Logistic Regression and XGBoost
print(f"Logistic Regression - Accept Rate: {accept_rate_lr:.2%}, Bad Rate: {bad_rate_lr:.2%}")
print(f"XGBoost - Accept Rate: {accept_rate_xgb:.2%}, Bad Rate: {bad_rate_xgb:.2%}")

## Random Forest

In [ ]:
# random forest
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)  

# Print classification report for Random Forest
print(classification_report(y_test, y_pred_rf, target_names=['Not Delinquent (0)', 'Delinquent (1)']))  

# confustion matrix for random forest
cm_rf = confusion_matrix(y_test, y_pred_rf)
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_rf,
    display_labels=['Not Delinquent (0)', 'Delinquent (1)']
)
fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(cmap='Blues', ax=ax, colorbar=False, values_format='d')
ax.set_title('Confusion Matrix for Random Forest\nRows: Actual, Columns: Predicted')
plt.tight_layout()
plt.show()  

# calculate accept rate and bad rate based on Random Forest predictions
accept_rate_rf, bad_rate_rf = calculate_rates(y_test, y_pred_rf)
print(f"Random Forest - Accept Rate: {accept_rate_rf:.2%}, Bad Rate: {bad_rate_rf:.2%}")

## Model Comparison

In [ ]:
# accept and bad rate coordinate plot with middle lines at 0.5 for both axes
def accept_bad_plot_base():
    fig, ax = plt.subplots(figsize=(9, 6), constrained_layout=True)
    ax.axhline(0.5, color='gray', linestyle='--', linewidth=1)
    ax.axvline(0.5, color='gray', linestyle='--', linewidth=1)
    ax.set_xlim(0, 1)
    ax.set_ylim(1,0)
    ax.set_xlabel('Accept Rate (0 to 1)')
    ax.set_ylabel('Bad Rate')
    ax.set_title('Accept Rate vs Bad Rate by Model', pad=14, loc='left')
    ax.grid(alpha=0.2)

    # Reset label-placement memory each time a new chart is created.
    add_model_point.placed = []
    return fig, ax


def add_model_point(ax, accept_rate, bad_rate, model_name):
    ax.scatter(accept_rate, bad_rate, s=70, label=model_name)

    # Auto choose label side based on edge proximity and nearby label collisions.
    side = 'left' if accept_rate > 0.82 else 'right'
    for px, py, pside in getattr(add_model_point, 'placed', []):
        close_x = abs(px - accept_rate) < 0.06
        close_y = abs(py - bad_rate) < 0.06
        if close_x and close_y and pside == side:
            side = 'left' if side == 'right' else 'right'
            break

    dx = 8 if side == 'right' else -8
    ha = 'left' if side == 'right' else 'right'

    ax.annotate(
        model_name,
        xy=(accept_rate, bad_rate),
        xytext=(dx, 6),
        textcoords='offset points',
        ha=ha,
        fontsize=9,
        bbox=dict(boxstyle='round,pad=0.2', fc='white', ec='none', alpha=0.7)
    )

    add_model_point.placed.append((accept_rate, bad_rate, side))

In [ ]:
# Baseline - accept all
y_pred_baseline = np.zeros_like(y_test)
accept_rate_baseline, bad_rate_baseline = calculate_rates(y_test, y_pred_baseline)
print(f"Baseline (Accept All) - Accept Rate: {accept_rate_baseline:.2%}, Bad Rate: {bad_rate_baseline:.2%}")



In [ ]:
# Accept vs bad rate plot for all models
fig, ax = accept_bad_plot_base()

add_model_point(ax, accept_rate_baseline, bad_rate_baseline, 'Baseline (Accept All)')
add_model_point(ax, accept_rate_lr, bad_rate_lr, 'Logistic Regression')
add_model_point(ax, accept_rate_xgb, bad_rate_xgb, 'XGBoost')
add_model_point(ax, accept_rate_rf, bad_rate_rf, 'Random Forest')

# Prefer lower bad rate for the same accept rate.
ax.invert_yaxis()
ax.set_ylabel('Bad Rate (lower is better)')
ax.set_xlim(0, 1)

ax.legend(loc='upper right', frameon=True)
plt.show()